In [1]:
# ==========================================================================
# Serving Project : 사내 문서 질의응답 서비스 (정리본)
#   흐름 : 브라우저 --(JSON + X-API-Key)--> FastAPI
#          : 인증 -> 부서 확인 -> 문서 검색 -> 생성 -> 스트리밍
#   서버 : app/main.py -> http://127.0.0.1:8452 (비어 있는 포트를 자동으로 찾음)
#   화면 : frontend/app.py -> 터미널에서 Streamlit 실행 (3.10)
#   키   : hr-key-003(인사), sales-key-002(영업), dev-key-001(개발), partner-key-004(공개만)
#   커널 : Python 3.12 (WSL model-serving) = .venv-wsl312
#   결과 : 채점 12/12 통과, 동시 4명까지 6초 안에 응답
# ==========================================================================
# [0. 패키지] 노트북이 검증한 버전으로 맞춥니다. 이미 깔려 있으면 아무 일도 하지 않습니다
# 패키지 설치. %pip 은 코랩과 로컬 주피터 모두 이 노트북을 돌리는 파이썬에 깝니다
%pip install -q "fastapi>=0.141,<0.142" "uvicorn[standard]>=0.53,<0.54" \
                "starlette>=1.6,<2" "pydantic>=2.13,<3" \
                "httpx>=0.28,<1" "anyio>=4.15,<5" "streamlit>=1.64,<2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# [0. 패키지] 버전 기록
# 실제로 깔린 버전을 찍어 둡니다. 결과가 달라졌을 때 가장 먼저 볼 기록입니다
from importlib.metadata import version, PackageNotFoundError

for name in ["fastapi", "uvicorn", "starlette", "pydantic",
             "httpx", "anyio", "streamlit"]:
    try:
        print(f"{name:12} {version(name)}")
    except PackageNotFoundError:
        print(f"{name:12} (설치되지 않음)")

import pydantic
print()
print("Pydantic 계열 :", "v2" if pydantic.VERSION.startswith("2")
      else "v1 -- 이 노트의 코드는 v1에서 돌지 않습니다")

fastapi      0.141.1
uvicorn      0.53.0
starlette    1.10.0
pydantic     2.14.2
httpx        0.28.1
anyio        4.16.0
streamlit    1.66.1



Pydantic 계열 : v2


In [3]:
# ==========================================================================
# [1. 환경 준비] 작업 폴더와 포트를 정합니다
#   - pick_port() : 8452 부터 올라가며 비어 있는 포트를 찾습니다
#   - 주소, 포트, 정책 값은 전부 환경 변수(D52_*)에서 읽습니다 (상수를 박지 않음)
#   - app/__init__.py 를 만들어 app 을 파이썬 패키지로 인식시킵니다
#     (없으면 uvicorn 이 app.xxx 를 못 찾음)
# ==========================================================================
import os
import socket
import sys
from pathlib import Path


def pick_port(base, tries=20):
    """base 부터 올라가며 비어 있는 포트를 찾습니다. 앞 노트와 같은 방식입니다."""
    for port in range(base, base + tries):
        with socket.socket() as probe:
            try:
                probe.bind(("127.0.0.1", port))
                return port
            except OSError:
                continue
    return base


# 상수를 박지 않습니다. 이 노트의 포트는 8452 이고, 막혀 있으면 위로 올라갑니다
PORT = pick_port(int(os.environ.get("D52_PORT", "8452")))
BASE = f"http://127.0.0.1:{PORT}"

WORK = Path(os.environ.get("D52_WORK", "."))
for d in ["app", "data", "frontend", "models"]:
    (WORK / d).mkdir(parents=True, exist_ok=True)

# app 을 파이썬 패키지로 인식시킵니다. 이 파일이 없으면 uvicorn 이 app.xxx 를 못 찾습니다
(WORK / "app" / "__init__.py").write_text("", encoding="utf-8")

0

In [4]:
# [1. 환경 준비] 작업 폴더를 import 경로에 넣고 서버 주소를 확인합니다
if str(WORK.resolve()) not in sys.path:
    sys.path.insert(0, str(WORK.resolve()))

print("작업 폴더 :", WORK.resolve())
print("서버 주소 :", BASE)

작업 폴더 : /mnt/c/EPA1st/model-serving-course
서버 주소 : http://127.0.0.1:8453


In [5]:
# [1. 서버 도구] 서버를 백그라운드 스레드로 띄우고 내리는 도구 (serve, unserve, ensure_qa)
#   같은 포트에 서버를 갈아 끼울 수 있습니다. 이 셀은 import 와 상태 변수
import importlib
import threading
import time

import uvicorn

_SERVER = None          # 지금 떠 있는 서버 한 대

In [6]:
# [1. 서버 도구] serve() : uvicorn 서버를 백그라운드 스레드에서 띄웁니다
#   - 이미 떠 있는 서버는 먼저 내림 -> 포트는 하나만 씀
#   - 같은 이름의 모듈(app.*)을 sys.modules 에서 비워 옛 코드가 남지 않게 함
def serve(target, wait=30.0):
    """uvicorn 서버를 백그라운드 스레드에서 띄웁니다.

    target 은 "app.main:app" 처럼 "모듈:변수" 형식입니다.
    이미 떠 있는 서버가 있으면 먼저 내립니다. 그래서 포트는 하나만 씁니다.
    """
    global _SERVER
    unserve()

    # 같은 이름의 모듈을 고쳐 다시 띄울 때 옛 내용이 남지 않도록 비웁니다
    for name in [n for n in list(sys.modules) if n == "app" or n.startswith("app.")]:
        del sys.modules[name]
    importlib.invalidate_caches()

    config = uvicorn.Config(target, host="127.0.0.1", port=PORT,
                            log_level="warning", access_log=False)
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    deadline = time.time() + wait
    while time.time() < deadline:
        if server.started:
            _SERVER = (server, thread)
            return f"기동 : {target} -> {BASE}"
        if not thread.is_alive():
            return f"기동 실패 : {target}. 위에 찍힌 오류를 보십시오"
        time.sleep(0.1)
    return f"기동 대기 {wait}초를 넘겼습니다"

In [7]:
# [1. 서버 도구] unserve() : 떠 있는 서버를 내립니다
def unserve(wait=30.0):
    """떠 있는 서버를 내립니다."""
    global _SERVER
    if _SERVER is None:
        return "띄운 서버가 없습니다"
    server, thread = _SERVER
    server.should_exit = True
    thread.join(timeout=wait)
    _SERVER = None
    return "정지" if not thread.is_alive() else "시간 안에 내려가지 않았습니다"


print(unserve())

띄운 서버가 없습니다


In [8]:
# ==========================================================================
# [2. 부품 점검] 앞 노트들에서 만든 파일(로깅, 에러 핸들러, 미들웨어, 인증, 문서)이 있는지 봅니다
# ==========================================================================
NEEDED = {
    "app/logger_config.py": "로깅 설정 (비동기 처리 노트)",
    "app/error_handlers.py": "글로벌 에러 핸들러 (비동기 처리 노트)",
    "app/middleware.py": "요청 로깅 미들웨어 (비동기 처리 노트)",
    "app/auth.py": "API 키 인증 (인증과 화면 노트)",
    "data/사내문서.json": "사내 문서 (데이터셋 노트)",
}

missing = [p for p in NEEDED if not (WORK / p).exists()]
for path, 설명 in NEEDED.items():
    표시 = "있음" if (WORK / path).exists() else "없음 -> 다음 셀이 만듭니다"
    print(f"  {path:28} {표시:26} {설명}")
print()
print(f"없는 부품 {len(missing)}개")

  app/logger_config.py         있음                         로깅 설정 (비동기 처리 노트)
  app/error_handlers.py        있음                         글로벌 에러 핸들러 (비동기 처리 노트)
  app/middleware.py            있음                         요청 로깅 미들웨어 (비동기 처리 노트)
  app/auth.py                  있음                         API 키 인증 (인증과 화면 노트)
  data/사내문서.json               있음                         사내 문서 (데이터셋 노트)

없는 부품 0개


In [9]:
# [2. 부품 재생성 1/4] 없는 부품을 같은 계약으로 다시 만듭니다 - 로깅 설정(app/logger_config.py) 내용
LOGGER_PY = '''"""로깅 설정. 비동기 처리 노트 5.3절의 것에 propagate = False를 더했습니다."""
import logging
import sys


def setup_logger(name="ml_api", level="INFO"):
    """콘솔 로거를 만듭니다. 여러 번 불러도 핸들러가 겹치지 않습니다."""
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level))
    if logger.handlers:
        return logger
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        fmt="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    ))
    logger.addHandler(handler)
    logger.propagate = False
    return logger
'''

In [10]:
# [2. 부품 재생성 2/4] 글로벌 에러 핸들러(app/error_handlers.py) 내용
#   상세는 로그에, 요청자에게는 한 줄만 (내부 정보가 새지 않게)
ERRORS_PY = '''"""글로벌 에러 핸들러. 상세는 로그에, 요청자에게는 한 줄."""
import logging
import traceback

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

logger = logging.getLogger("ml_api")


def register_error_handlers(app: FastAPI):
    @app.exception_handler(Exception)
    async def unhandled(request: Request, exc: Exception):
        logger.error(
            f"처리하지 못한 예외 : {type(exc).__name__}: {exc}\\n"
            f"  경로 : {request.method} {request.url.path}\\n"
            f"{traceback.format_exc()}"
        )
        return JSONResponse(
            status_code=500,
            content={"success": False, "error": "서버 내부 오류가 발생했습니다"},
        )
'''

In [11]:
# [2. 부품 재생성 3/4] 요청 로깅 미들웨어(app/middleware.py) 내용 - 순수 ASGI
MIDDLEWARE_PY = '''"""요청 로깅 미들웨어 (순수 ASGI).

BaseHTTPMiddleware 를 쓰지 않는 이유는 비동기 처리 노트에 적어 두었습니다.
스트리밍 도중 사용자가 떠날 때 정리 코드가 제때 돌지 않습니다.
"""
import logging
import time

logger = logging.getLogger("ml_api")


class RequestLoggingMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return

        started = time.perf_counter()
        info = {"status": 0, "머리까지": None, "완료": False}

        async def send_and_watch(message):
            if message["type"] == "http.response.start":
                info["status"] = message["status"]
                info["머리까지"] = time.perf_counter() - started
                message["headers"].append(
                    (b"x-process-time", f"{info['머리까지']:.3f}".encode()))
            elif (message["type"] == "http.response.body"
                  and not message.get("more_body", False)):
                info["완료"] = True
            await send(message)

        label = f"{scope['method']} {scope['path']}"
        failed = None
        try:
            await self.app(scope, receive, send_and_watch)
        except Exception as exc:
            failed = type(exc).__name__
            raise
        finally:
            total = time.perf_counter() - started
            if failed:
                logger.error(f"{label} -> 예외 {failed} ({total:.2f}초)")
            elif info["머리까지"] is None:
                logger.warning(f"{label} -> 응답을 시작하지 못했습니다 ({total:.2f}초)")
            else:
                끝 = "완료" if info["완료"] else "중간에 끊김"
                line = (f"{label} -> {info['status']} "
                        f"(머리까지 {info['머리까지']:.2f}초, "
                        f"끝까지 {total:.2f}초, {끝})")
                if info["status"] >= 400 or not info["완료"]:
                    logger.warning(line)
                else:
                    logger.info(line)
'''

In [12]:
# [2. 부품 재생성 4/4] 위의 세 내용을 파일로 저장합니다
for 이름, 내용 in [("app/logger_config.py", LOGGER_PY),
                 ("app/error_handlers.py", ERRORS_PY),
                 ("app/middleware.py", MIDDLEWARE_PY)]:
    (WORK / 이름).write_text(내용, encoding="utf-8")
    print(f"  {이름:28} {len(내용):5}자")

  app/logger_config.py           588자
  app/error_handlers.py          659자
  app/middleware.py             1944자


In [13]:
%%writefile app/auth.py
# [2.1 부서까지 돌려주는 인증 app/auth.py (1/3)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   "dev-key-001" -> {"사용자": "김개발", "부서": ["공개", "개발"]}
#   키가 없거나 틀리면 401. 챗봇 부록도 이 파일을 함께 씁니다
"""API 키 인증. 인증과 화면 노트에서 만든 것에 부서 필드 하나를 더했습니다.

진짜 사내 서비스에서는 이 표가 사내 계정 시스템(LDAP, SSO)에서 옵니다.
키를 코드에 적어 두는 것은 실습에서만 하는 일이고, 사내에 올릴 때는
환경 변수나 시크릿 저장소에서 읽습니다.
"""
import json
import os

from fastapi import Header, HTTPException

Overwriting app/auth.py


In [14]:
%%writefile -a app/auth.py

# [app/auth.py (2/3)] API 키 -> 사용자, 부서 표
# 기본값은 실습용입니다. 사내에서는 D52_API_KEYS 환경 변수로 덮어씁니다
_DEFAULT_KEYS = {
    "dev-key-001": {"사용자": "김개발", "부서": ["공개", "개발"]},
    "sales-key-002": {"사용자": "이영업", "부서": ["공개", "영업"]},
    "hr-key-003": {"사용자": "박인사", "부서": ["공개", "인사"]},
    "partner-key-004": {"사용자": "협력사 계정", "부서": ["공개"]},
}

API_KEYS = json.loads(os.environ["D52_API_KEYS"]) if os.environ.get("D52_API_KEYS") \
    else _DEFAULT_KEYS

Appending to app/auth.py


In [15]:
%%writefile -a app/auth.py

# [app/auth.py (3/3)] verify_api_key() : 헤더의 API 키를 확인하고 사용자 정보(이름, 부서)를 돌려줍니다
async def verify_api_key(x_api_key: str = Header(None)) -> dict:
    """헤더의 API 키를 확인하고 사용자 정보를 돌려줍니다.

    돌려주는 것 : {"사용자": 이름, "부서": [열람 가능한 부서 목록]}
    """
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API 키가 필요합니다. X-API-Key 헤더를 포함해 주십시오.",
        )
    if x_api_key not in API_KEYS:
        raise HTTPException(status_code=401, detail="유효하지 않은 API 키입니다.")
    return API_KEYS[x_api_key]

Appending to app/auth.py


In [16]:
# [2.2 사내 문서에 부서 붙이기 data/사내문서.json (1/2)]
#   문서마다 '부서' 필드가 있어야 열람 범위를 정할 수 있습니다
import json

사내문서 = [
    {"제목": "재택근무 안내", "부서": "공개",
     "본문": "재택근무는 주 2일까지 신청할 수 있고 팀장 승인이 필요합니다. "
             "신청은 전날 정오까지 사내 포털에서 합니다."},
    {"제목": "연차 규정", "부서": "공개",
     "본문": "연차는 입사 1년 차에 11일, 2년 차부터 15일이 부여됩니다. "
             "미사용 연차는 다음 해 3월까지 쓸 수 있습니다."},
    {"제목": "장비 반출 절차", "부서": "공개",
     "본문": "노트북을 외부로 가지고 나갈 때는 보안팀 사전 승인이 필요합니다. "
             "승인 없이 반출하면 감사 대상이 됩니다."},
    {"제목": "인사 평가 등급 기준", "부서": "인사",
     "본문": "평가 등급은 S, A, B, C 네 단계이며 S 등급은 팀별 상위 10퍼센트로 제한합니다. "
             "등급별 성과급 지급률은 인사위원회가 매년 정합니다."},
    {"제목": "신입 연봉 테이블", "부서": "인사",
     "본문": "신입 초임은 직군별로 다르며 개발 직군이 가장 높습니다. "
             "연봉 정보는 본인 외에 공유할 수 없습니다."},
    {"제목": "4분기 영업 목표", "부서": "영업",
     "본문": "4분기 목표는 3분기 실적의 120퍼센트이며 신규 계약 비중을 절반 이상으로 합니다. "
             "주간 실적은 매주 월요일에 집계합니다."},
    {"제목": "개발 서버 접근 절차", "부서": "개발",
     "본문": "개발 서버는 사내망에서만 접속할 수 있고 계정은 팀장이 신청합니다. "
             "운영 데이터베이스 직접 접근은 금지되어 있습니다."},
]

DOC_PATH = WORK / "data" / "사내문서.json"
DOC_PATH.write_text(json.dumps(사내문서, ensure_ascii=False, indent=1), encoding="utf-8")

부서별 = {}

In [17]:
# [2.2 (2/2)] 부서별 문서 수를 세어 확인합니다
for d in 사내문서:
    부서별[d["부서"]] = 부서별.get(d["부서"], 0) + 1

print(f"문서 {len(사내문서)}건 저장 : {DOC_PATH}")
for 부서, 수 in sorted(부서별.items()):
    print(f"  {부서:6} {수}건")

문서 7건 저장 : data/사내문서.json
  개발     1건
  공개     3건
  영업     1건
  인사     2건


In [18]:
%%writefile app/inference.py
# ==========================================================================
# [3. 사내 문서 질의응답 서비스]
# [3.1 검색과 생성 app/inference.py (1/4)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   - retrieve(question, 부서) : 맨 먼저 부서로 거른 뒤 검색 (오늘의 핵심 한 줄)
#   - generate_tokens() : 속도를 통제하려고 mock. 진짜 모델로 바꾸는 자리는 여기 한 곳
#   - 이 셀 : load_docs(), 생성 속도(TPS), 토큰당 글자 수, 근거 채택 최소 겹침 수(MIN_HITS)
#     MIN_HITS 를 낮추면 엉뚱한 문서가 붙고, 높이면 답을 못 합니다
# ==========================================================================
"""사내 문서 질의응답 : 검색과 생성.

진짜 서비스에서는 사내 RAG 노트에서 만든 벡터 검색기와, 사내 데이터로 학습한
어댑터를 붙인 모델이 여기 들어갑니다. 오늘의 주제는 서버 조립이라 생성부만
mock을 씁니다. 바꿔 끼우는 자리는 generate_tokens() 하나입니다.
"""
import json
import os
import time
from pathlib import Path

# 생성 속도. 진짜 모델이라면 GPU와 모델 크기가 정하는 값입니다
TOKENS_PER_SECOND = float(os.environ.get("D52_TPS", "20"))

# 한 토큰에 몇 글자로 볼 것인가. 한국어는 보통 한두 글자가 한 토큰입니다
CHARS_PER_TOKEN = 3

DOC_PATH = Path(os.environ.get("D52_DOCS", "data/사내문서.json"))

# 근거로 채택할 최소 겹침 수. 낮추면 엉뚱한 문서가 붙고 높이면 답을 못 합니다
MIN_HITS = int(os.environ.get("D52_MIN_HITS", "3"))


def load_docs():
    """사내 문서를 읽습니다."""
    return json.loads(DOC_PATH.read_text(encoding="utf-8"))

Overwriting app/inference.py


In [19]:
%%writefile -a app/inference.py

# [app/inference.py (2/4)]
#   - _grams()   : 두 글자씩 잘라 집합으로 만듭니다
#   - retrieve() : 볼 수 있는 부서의 문서 중에서 질문과 겹치는 청크를 고릅니다
def _grams(text, n=2):
    """두 글자씩 잘라 집합으로 만듭니다."""
    t = "".join(text.split())
    return {t[i:i + n] for i in range(len(t) - n + 1)}


def retrieve(question, allowed_depts, k=2):
    """볼 수 있는 부서의 문서 중에서 질문과 겹치는 청크를 고릅니다.

    부서로 거르는 일이 점수를 매기는 일보다 **먼저** 옵니다.
    순서가 뒤집히면 볼 수 없는 문서가 프롬프트에 들어갑니다.

    진짜 서비스에서는 이 자리에 사내 RAG 노트의 벡터 검색이 들어가고,
    부서 조건은 벡터DB 의 메타데이터 필터가 됩니다.
    """
    docs = [d for d in load_docs() if d["부서"] in allowed_depts]
    q = _grams(question)
    scored = []
    for d in docs:
        hits = len(q & _grams(d["제목"] + " " + d["본문"]))
        if hits >= MIN_HITS:
            scored.append((hits, d))
    scored.sort(key=lambda x: -x[0])
    return [d for _, d in scored[:k]]

Appending to app/inference.py


In [20]:
%%writefile -a app/inference.py

# [app/inference.py (3/4)] _compose() : 근거를 붙여 답변 문장을 만듭니다
#   근거가 없으면 예외 대신 "근거를 못 찾았다" 는 답변(200)으로 돌려줍니다
def _compose(question, docs):
    """근거를 붙여 답변 문장을 만듭니다."""
    if not docs:
        # 실패를 예외가 아니라 답변으로 다룹니다. 서버는 200 을 돌려주고,
        # 직원은 "권한이 없다" 가 아니라 "근거를 못 찾았다" 를 읽습니다
        return ("문의하신 내용은 지금 계정으로 열람할 수 있는 사내 문서에서 "
                "근거를 찾지 못했습니다. 문서 담당 부서에 열람 권한을 요청하시거나 "
                "질문을 더 구체적으로 적어 다시 문의해 주십시오.")
    근거 = " ".join(d["본문"] for d in docs)
    return (f"문의하신 '{question}'에 대해 사내 규정을 확인했습니다. {근거} "
            f"자세한 내용은 담당 부서에 문의하시기 바랍니다.")

Appending to app/inference.py


In [21]:
%%writefile -a app/inference.py

# [app/inference.py (4/4)]
#   - generate_tokens() : 답변을 토큰 하나씩 내놓는 동기 제너레이터 (진짜 모델로 바꾸는 자리)
#   - generate()        : 답변 전체를 한 번에 돌려줍니다 (model.generate() 자리)
def generate_tokens(question, docs):
    """답변을 토큰 하나씩 내놓는 동기 제너레이터.

    ===== 진짜 모델로 바꾸는 자리는 여기 하나입니다 =====
    transformers 의 TextIteratorStreamer 를 쓰면 model.generate 를 별도
    스레드에서 돌리면서 이 함수와 똑같이 토큰을 하나씩 받아올 수 있습니다.
    바깥에서 보는 모양(동기 제너레이터, next() 한 번이 토큰 하나)은 같습니다.
    프롬프트에는 docs 의 본문만 넣습니다. 부서 확인은 이미 끝나 있습니다.
    """
    text = _compose(question, docs)
    for i in range(0, len(text), CHARS_PER_TOKEN):
        time.sleep(1.0 / TOKENS_PER_SECOND)
        yield text[i:i + CHARS_PER_TOKEN]


def generate(question, docs):
    """답변 전체를 한 번에 돌려줍니다. model.generate() 자리입니다."""
    return "".join(generate_tokens(question, docs))

Appending to app/inference.py


In [22]:
# [3.2 부서가 실제로 갈리는지 확인] 같은 질문이라도 인사 계정만 인사 문서를 봅니다
import importlib

import app.inference as inference
importlib.reload(inference)

질문 = "인사 평가 등급은 어떻게 정합니까"

for 계정, 부서 in [("이영업", ["공개", "영업"]),
                 ("박인사", ["공개", "인사"]),
                 ("협력사 계정", ["공개"])]:
    찾은것 = inference.retrieve(질문, 부서)
    제목들 = [d["제목"] for d in 찾은것] or ["(없음)"]
    print(f"  {계정:8} 열람 {str(부서):20} -> {', '.join(제목들)}")

  이영업      열람 ['공개', '영업']         -> (없음)
  박인사      열람 ['공개', '인사']         -> 인사 평가 등급 기준
  협력사 계정   열람 ['공개']               -> (없음)


In [23]:
# [3.2] 생성 시간 측정
# 생성 시간을 재 둡니다. 3.8절 부하 테스트의 기준선입니다
import time

근거 = inference.retrieve("재택근무는 며칠까지 가능합니까", ["공개"])
t0 = time.perf_counter()
토큰들 = list(inference.generate_tokens("재택근무는 며칠까지 가능합니까", 근거))
걸린시간 = time.perf_counter() - t0

print(f"근거      : {[d['제목'] for d in 근거]}")
print(f"답변      : {''.join(토큰들)[:60]}...")
print(f"토큰 수   : {len(토큰들)}개")
print(f"생성 시간 : {걸린시간:.2f}초  ({inference.TOKENS_PER_SECOND:.0f} 토큰/초)")
print()
print(f"이 노트에서 '답 하나'는 약 {걸린시간:.0f}초입니다. 이 숫자를 기억해 두십시오")

근거      : ['재택근무 안내']
답변      : 문의하신 '재택근무는 며칠까지 가능합니까'에 대해 사내 규정을 확인했습니다. 재택근무는 주 2일까지 신청할 ...
토큰 수   : 44개
생성 시간 : 2.21초  (20 토큰/초)

이 노트에서 '답 하나'는 약 2초입니다. 이 숫자를 기억해 두십시오


In [24]:
%%writefile app/generation.py
# [3.3 생성 층 app/generation.py (1/3)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   - CapacityLimiter(GPU_SLOTS) : GPU 한 장에 동시에 몇 건까지 통과시킬지
#   - 토큰 예산, 시간 예산       : 한 요청이 서버를 오래 잡지 않게
#   - request.is_disconnected()  : 떠난 사용자를 위해 생성하지 않게
#   - 상수를 박지 않습니다. 사내 서버마다 GPU도 정책도 다릅니다
"""생성 층 : 예산과 이탈을 보면서 토큰을 내놓습니다.

비동기 처리 노트 7.2절의 guarded_tokens 를 그대로 가져왔습니다.
달라진 것은 둘입니다. 검색 결과(docs)를 밖에서 받고, 마지막 생성 결과를
STATS 에 남깁니다. STATS 는 운영 중에 "방금 무슨 일이 있었나" 를 보는 창입니다.
"""
import json
import logging
import os
import time

import anyio

from app.inference import generate_tokens

logger = logging.getLogger("ml_api")

# 상수를 박지 않습니다. 사내 서버마다 GPU도 정책도 다릅니다
MAX_TOKENS = int(os.environ.get("D52_MAX_TOKENS", "500"))
WALL_CLOCK = float(os.environ.get("D52_WALL_CLOCK", "30"))
CHECK_EVERY = int(os.environ.get("D52_CHECK_EVERY", "10"))
GPU_SLOTS = int(os.environ.get("D52_GPU_SLOTS", "2"))

gpu_limiter = anyio.CapacityLimiter(GPU_SLOTS)

STATS = {"동시 통과": GPU_SLOTS, "누적 요청": 0, "누적 토큰": 0, "마지막": None}

Overwriting app/generation.py


In [25]:
%%writefile -a app/generation.py

# [app/generation.py (2/3)]
#   - new_report()     : 생성 결과 기록(정지 사유, 토큰 수, 대기/생성 시간)
#   - new_budget()     : 요청이 값을 주지 않으면 서버 기본값을 씁니다
#   - sse()            : Server-Sent Events 한 청크 형식
#   - guarded_tokens() : 예산과 이탈을 보며 토큰을 내놓고, 정지 사유를 report 에 남깁니다
def new_report():
    return {"정지 사유": "진행 중", "토큰": 0, "대기 초": 0.0, "생성 초": 0.0}


def new_budget(max_tokens=None, deadline_seconds=None):
    """요청이 값을 주지 않으면 서버 기본값을 씁니다."""
    return {"max_tokens": max_tokens or MAX_TOKENS,
            "deadline": deadline_seconds or WALL_CLOCK}


def sse(event, data):
    """Server-Sent Events wire format 한 청크."""
    return f"event: {event}\ndata: {json.dumps(data, ensure_ascii=False)}\n\n"


async def guarded_tokens(question, docs, request, budget, report):
    """예산과 이탈을 보며 토큰을 내놓습니다. 정지 사유는 report 에 남깁니다."""
    it = generate_tokens(question, docs)
    done = object()
    queued_at = time.perf_counter()
    started = queued_at
    count = 0
    report["정지 사유"] = "진행 중"

Appending to app/generation.py


In [26]:
%%writefile -a app/generation.py

# [app/generation.py (3/3)] guarded_tokens() 이어서
#   문지기(gpu_limiter) : 동시에 GPU_SLOTS 건만 통과하고 나머지는 기다립니다
    try:
        # 문지기. 동시에 GPU_SLOTS 건만 통과하고 나머지는 여기서 기다립니다
        async with gpu_limiter:
            started = time.perf_counter()
            report["대기 초"] = round(started - queued_at, 2)
            while True:
                token = await anyio.to_thread.run_sync(next, it, done)
                if token is done:
                    report["정지 사유"] = "완료"
                    break
                count += 1
                yield token
                if count >= budget["max_tokens"]:
                    report["정지 사유"] = "토큰 예산 소진"
                    break
                if time.perf_counter() - started >= budget["deadline"]:
                    report["정지 사유"] = "시간 예산 소진"
                    break
                if count % CHECK_EVERY == 0 and await request.is_disconnected():
                    report["정지 사유"] = "클라이언트 이탈"
                    break
    finally:
        it.close()                      # 여기가 GPU를 아끼는 한 줄입니다
        if report["정지 사유"] == "진행 중":
            report["정지 사유"] = "밖에서 끊김"
        report["토큰"] = count
        report["생성 초"] = round(time.perf_counter() - started, 2)
        STATS["누적 요청"] += 1
        STATS["누적 토큰"] += count
        STATS["마지막"] = dict(report)
        logger.info(f"생성 종료 : {report}")

Appending to app/generation.py


In [27]:
%%writefile app/schemas.py
# [3.4 요청/응답 규격 app/schemas.py (1/3)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   질문은 1~500자. 빈 질문이나 너무 긴 질문은 422
#   입력 길이가 곧 비용이라 상한을 둡니다
# 실습 1 : 요청과 응답의 규격 (AskRequest, Source, AskResponse)
"""요청과 응답의 규격.

참고 : FastAPI 기초 노트의 Request Body 절과 문서를 친절하게 만드는 절.
"""
from pydantic import BaseModel, Field

Overwriting app/schemas.py


In [28]:
%%writefile -a app/schemas.py

# [app/schemas.py (2/3)] AskRequest : 질의 요청
#   question(필수, 1~500자) / max_tokens(선택, 1~2000) / deadline_seconds(선택, 0 초과 300 이하)
class AskRequest(BaseModel):
    """질의 요청."""
    # 1) question : 필수. 1자 이상 500자 이하 (통과 기준 7번과 8번)
    question: str = Field(
        ..., min_length=1, max_length=500,
        description="사내 규정에 대한 질문. 길수록 비쌉니다",
    )
    # 2) max_tokens : 선택. 1 이상 2000 이하. 기본값 None
    max_tokens: int | None = Field(
        default=None, ge=1, le=2000,
        description="이 요청에만 적용할 토큰 예산. 없으면 서버 기본값",
    )
    # 3) deadline_seconds : 선택. 0 초과 300 이하. 기본값 None
    deadline_seconds: float | None = Field(
        default=None, gt=0, le=300,
        description="이 요청에만 적용할 시간 예산(초). 없으면 서버 기본값",
    )

    model_config = {
        "json_schema_extra": {
            "examples": [
                {"question": "재택근무는 며칠까지 가능합니까",
                 "max_tokens": 200,
                 "deadline_seconds": 20.0}
            ]
        }
    }

Appending to app/schemas.py


In [29]:
%%writefile -a app/schemas.py

# [app/schemas.py (3/3)] Source : 근거 문서 한 건(제목, 부서) / AskResponse : 질의 응답
class Source(BaseModel):
    """근거 문서 한 건."""
    # 1) 제목 : 사내 문서 제목
    제목: str = Field(description="사내 문서 제목")
    # 2) 부서 : 그 문서를 소유한 부서
    부서: str = Field(description="그 문서를 소유한 부서")


class AskResponse(BaseModel):
    """질의 응답."""
    success: bool = Field(default=True, description="요청 처리 성공 여부")
    answer: str = Field(description="생성된 답변")
    sources: list[Source] = Field(default_factory=list, description="근거로 쓴 문서")
    user: str = Field(description="인증된 사용자 이름")
    report: dict = Field(default_factory=dict, description="정지 사유, 토큰 수, 대기 초, 생성 초")

Appending to app/schemas.py


In [30]:
%%writefile app/service.py
# [3.5 서비스 층 app/service.py (1/2)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   누가 무엇을 볼 수 있는지 한 곳에서만 정합니다
#   부서를 모르면 아무것도 보여 주지 않습니다 ("모르면 불합격")
# 실습 2 : 서비스 층 (resolve_docs, to_sources, in_scope)
"""서비스 층 : 누가 무엇을 볼 수 있는지 정합니다.

참고 : 인증과 화면 노트의 Depends 절, 사내 RAG 노트의 검색 절.

판정 함수는 "모르면 불합격" 으로 만듭니다. 그 반대로 만들면
버그가 났을 때 조용히 통과합니다.
"""
from app.inference import retrieve

Overwriting app/service.py


In [31]:
%%writefile -a app/service.py

# [app/service.py (2/2)]
#   - resolve_docs() : 사용자의 부서 목록으로 retrieve -> 부서가 없으면 빈 목록(안전한 쪽)
#   - to_sources()   : 응답에 실을 근거 목록. 본문은 빼고 제목과 부서만
#   - in_scope()     : 고른 문서가 전부 열람 범위 안인지 확인. 기본값은 불합격
def resolve_docs(question, user):
    """사용자가 볼 수 있는 범위에서 근거 문서를 고릅니다."""
    # 1) user 의 부서 목록. user 가 None 이거나 부서 필드가 없으면 빈 목록
    부서 = (user or {}).get("부서") or []
    # 2) 부서 필드가 없는 user 는 빈 목록 (안전한 쪽)
    if not 부서:
        return []                      # 부서를 모르면 아무것도 보여주지 않습니다
    # 3) 부서 목록을 retrieve 에 넘겨 근거 문서를 받습니다
    return retrieve(question, 부서)


def to_sources(docs):
    """응답에 실을 근거 목록으로 바꿉니다. 본문은 빼고 제목과 부서만."""
    # 1) 본문은 넣지 않습니다. 응답이 무거워지고, 볼 필요도 없습니다
    return [{"제목": d["제목"], "부서": d["부서"]} for d in docs]


def in_scope(docs, user):
    """고른 문서가 전부 열람 범위 안인지 확인합니다. 기본값은 불합격입니다."""
    부서 = (user or {}).get("부서") or []
    if not 부서:
        return False
    # 1) 모든 문서의 부서가 user 의 부서 목록 안에 있는지
    return all(d.get("부서") in 부서 for d in docs)

Appending to app/service.py


In [32]:
# [3.5] 서버 없이 서비스 층만 확인합니다
#   부서 정보가 깨진 계정은 근거 0건, 판정 False 가 나와야 합니다
# 서버를 띄우지 않고 서비스 층만 확인합니다
import app.service as service
importlib.reload(service)

for 키, 사용자 in [("sales-key-002", {"사용자": "이영업", "부서": ["공개", "영업"]}),
                 ("hr-key-003", {"사용자": "박인사", "부서": ["공개", "인사"]}),
                 ("(부서 없음)", {"사용자": "깨진 계정"})]:
    docs = service.resolve_docs("인사 평가 등급은 어떻게 정합니까", 사용자)
    sources = service.to_sources(docs)
    print(f"  {사용자['사용자']:8} 근거 {len(sources)}건 {[s['제목'] for s in sources]}"
          f"  범위 안 : {service.in_scope(docs, 사용자)}")

  이영업      근거 0건 []  범위 안 : True
  박인사      근거 1건 ['인사 평가 등급 기준']  범위 안 : True
  깨진 계정    근거 0건 []  범위 안 : False


In [33]:
%%writefile app/main.py
# [3.6 서버 조립 app/main.py (1/7)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   엔드포인트 : /health, /stats, /ask(한 번에 답), /ask/sse(스트리밍), /ask/broken(일부러 500)
# 실습 3 : 서버 조립 (POST /ask, POST /ask/sse)
"""사내 문서 질의응답 서버.

참고 : FastAPI 기초 노트의 경로 장, 비동기 처리 노트 7장의 스트리밍 서버.
"""
import os

from contextlib import aclosing, asynccontextmanager

from fastapi import Depends, FastAPI, Request
from fastapi.responses import StreamingResponse

from app.auth import verify_api_key
from app.error_handlers import register_error_handlers
from app.generation import STATS, guarded_tokens, new_budget, new_report, sse
from app.inference import load_docs
from app.logger_config import setup_logger
from app.middleware import RequestLoggingMiddleware
from app.schemas import AskRequest, AskResponse
from app.service import in_scope, resolve_docs, to_sources

logger = setup_logger("ml_api")

STATE = {"준비됨": False, "문서 수": 0}

Overwriting app/main.py


In [34]:
%%writefile -a app/main.py

# [app/main.py (2/7)] lifespan() : 서버가 뜰 때 문서를 읽고 준비 상태를 표시
#   진짜 서비스라면 여기서 모델과 어댑터를 올립니다
@asynccontextmanager
async def lifespan(app: FastAPI):
    # 뜰 때. 진짜 서비스라면 여기서 모델과 어댑터를 올립니다
    logger.info("사내 문서를 읽습니다")
    STATE["문서 수"] = len(load_docs())
    STATE["준비됨"] = True
    logger.info(f"준비 완료. 문서 {STATE['문서 수']}건")
    yield
    STATE["준비됨"] = False
    logger.info("서버를 내립니다")

Appending to app/main.py


In [35]:
%%writefile -a app/main.py

# [app/main.py (3/7)] FastAPI 앱 생성, 미들웨어와 에러 핸들러 등록
#   사내 서버에서는 문서(/docs) 공개 범위를 환경 변수로 닫습니다
app = FastAPI(
    title="사내 문서 질의응답",
    description="사내망 안에서만 도는 질의응답 서비스입니다. 부서별로 열람 범위가 다릅니다.",
    version="1.0.0",
    lifespan=lifespan,
    # 사내 서버에서는 문서 공개 범위를 환경 변수로 닫습니다
    docs_url=os.environ.get("D52_DOCS_URL", "/docs"),
)
app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

Appending to app/main.py


In [36]:
%%writefile -a app/main.py

# [app/main.py (4/7)]
#   - _prepare() : resolve_docs 로 근거를 고르고 in_scope 로 한 번 더 확인
#                  범위 밖이면 500 대신 근거를 비우고 기록만 남김 (요청자를 탓하지 않음)
#   - GET /health : 게이트웨이가 부르는 경로. 절대 무거운 일을 하지 않습니다
#   - GET /stats  : 운영 지표. 방금 무슨 일이 있었는지 봅니다
def _prepare(req, user):
    """근거를 고르고 마지막으로 한 번 더 확인합니다."""
    # 1) resolve_docs 로 열람 범위 안에서 근거 문서를 고릅니다
    docs = resolve_docs(req.question, user)
    # 2) in_scope 로 한 번 더 확인합니다. 범위 밖이면 500 을 내지 말고 근거를 비운 채 진행합니다
    #    (권한 문제를 사용자 잘못으로 만들지 않습니다)
    if not in_scope(docs, user):
        # 여기까지 왔다는 것은 우리 코드에 구멍이 있다는 뜻입니다.
        # 요청자를 탓하지 말고, 근거를 비우고 기록을 남깁니다
        logger.error(f"열람 범위 밖 문서가 검색되었습니다 : user={user.get('사용자')}")
        docs = []
    return docs, new_budget(req.max_tokens, req.deadline_seconds)


@app.get("/health", tags=["System"])
async def health():
    """게이트웨이가 부르는 경로. 절대 무거운 일을 하지 않습니다."""
    return {"status": "healthy" if STATE["준비됨"] else "loading", **STATE}


@app.get("/stats", tags=["System"])
async def stats():
    """운영 지표. 방금 무슨 일이 있었는지 봅니다."""
    return STATS

Appending to app/main.py


In [37]:
%%writefile -a app/main.py

# [app/main.py (5/7)] POST /ask : 단일 응답. 답을 한 번에 돌려줍니다
#   aclosing 으로 감싸야 안쪽 제너레이터가 제때 닫힙니다. 토큰을 parts 에 모아 합칩니다
@app.post("/ask", response_model=AskResponse, tags=["Inference"])
async def ask(req: AskRequest, request: Request,
              user: dict = Depends(verify_api_key)):
    """단일 응답. 답을 한 번에 돌려줍니다."""
    docs, budget = _prepare(req, user)
    report = new_report()
    parts = []
    # 1) aclosing 으로 감쌉니다. 감싸지 않으면 안쪽 제너레이터가
    #    자동으로 닫히지 않습니다 (비동기 처리 노트 7.5절)
    async with aclosing(
            guarded_tokens(req.question, docs, request, budget, report)) as stream:
        # 2) 토큰을 하나씩 받아 parts 에 모읍니다
        async for token in stream:
            parts.append(token)
    return AskResponse(
        success=True,
        answer="".join(parts),
        sources=to_sources(docs),
        user=user["사용자"],
        report=report,
    )

Appending to app/main.py


In [38]:
%%writefile -a app/main.py

# [app/main.py (6/7)] POST /ask/sse : 스트리밍 응답. 만들어지는 대로 내보냅니다
#   스트리밍 도중의 예외는 상태 코드로 알릴 수 없습니다 (이미 200 이 나갔기 때문)
@app.post("/ask/sse", tags=["Inference"])
async def ask_sse(req: AskRequest, request: Request,
                  user: dict = Depends(verify_api_key)):
    """스트리밍 응답. 만들어지는 대로 내보냅니다."""
    docs, budget = _prepare(req, user)
    report = new_report()

    async def events():
        yield sse("start", {"question": req.question,
                            "sources": to_sources(docs),
                            "user": user["사용자"],
                            "예산": budget})
        try:
            async with aclosing(
                    guarded_tokens(req.question, docs, request, budget, report)) as stream:
                async for token in stream:
                    yield sse("token", {"text": token})
        except Exception as exc:
            logger.error(f"스트리밍 중 오류 : {type(exc).__name__}: {exc}")
            # 1) 스트리밍 도중의 예외는 상태 코드로 알릴 수 없습니다. 이미 200 이 나갔기 때문입니다
            yield sse("error", {"error": "생성 중 오류가 발생했습니다"})
            return
        yield sse("done", report)

    return StreamingResponse(
        events(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )

Appending to app/main.py


In [39]:
%%writefile -a app/main.py

# [app/main.py (7/7)] POST /ask/broken : 일부러 터뜨려 글로벌 에러 핸들러가 받는지 확인
@app.post("/ask/broken", tags=["System"])
async def ask_broken(req: AskRequest, user: dict = Depends(verify_api_key)):
    """일부러 터뜨립니다. 글로벌 핸들러가 받는지 확인하는 용도입니다."""
    raise RuntimeError(f"모델 가중치를 읽지 못했습니다 (질문 : {req.question})")

Appending to app/main.py


In [40]:
# [3.7 서버 띄우고 확인] app/main.py 를 띄우고 짧은 질의로 응답을 확인합니다
#   ensure_qa() : 서버가 내려가 있으면 다시 띄웁니다 (채점기/부하 테스트용)
import httpx

# 여러분이 채운 app/main.py 를 띄웁니다. 기동 실패를 예외로 만들지 않습니다
TARGET = os.environ.get("D52_TARGET", "app.main:app")
print(serve(TARGET))

시험 = httpx.post(BASE + "/ask", json={"question": "연차는 며칠입니까", "max_tokens": 5},
                headers={"X-API-Key": "dev-key-001"}, timeout=60)
if 시험.status_code != 200:
    print(f"  {TARGET} 이 아직 답하지 못합니다 (HTTP {시험.status_code}). "
          f"3.6절 실습 셀의 빈칸을 확인하십시오")
else:
    print("  여러분이 채운 서버가 답했습니다")


def ensure_qa():
    """질의응답 서버가 내려가 있으면(4장 선택 트랙에서 돌아왔을 때 등) 다시 띄웁니다. 채점기와 부하 테스트 셀이 ConnectError 로 멈추지 않게 합니다"""
    try:
        if "문서 수" in httpx.get(BASE + "/health", timeout=5).json():
            return
    except Exception:
        pass
    print(f"질의응답 서버 자동 재기동 : {serve(TARGET)}")

2026-11-04 14:41:36 INFO     [ml_api] 사내 문서를 읽습니다


2026-11-04 14:41:36 INFO     [ml_api] 준비 완료. 문서 7건


기동 : app.main:app -> http://127.0.0.1:8453


2026-11-04 14:41:36 INFO     [ml_api] 생성 종료 : {'정지 사유': '토큰 예산 소진', '토큰': 5, '대기 초': 0.0, '생성 초': 0.25}


2026-11-04 14:41:36 INFO     [ml_api] POST /ask -> 200 (머리까지 0.26초, 끝까지 0.26초, 완료)


  여러분이 채운 서버가 답했습니다


In [41]:
# [3.7] 헬스체크
print(httpx.get(BASE + "/health", timeout=10).json())

2026-11-04 14:41:37 INFO     [ml_api] GET /health -> 200 (머리까지 0.00초, 끝까지 0.00초, 완료)
{'status': 'healthy', '준비됨': True, '문서 수': 7}


In [42]:
# [3.7] 같은 질문을 인사/영업 계정으로 불러 봅니다
#   영업 계정에는 401/403 이 아니라 200 + 근거 없음 (문서가 있다는 사실도 숨김)
질문 = "인사 평가 등급은 어떻게 정합니까"

for 키 in ["hr-key-003", "sales-key-002"]:
    r = httpx.post(BASE + "/ask",
                   json={"question": 질문, "max_tokens": 40},
                   headers={"X-API-Key": 키}, timeout=120).json()
    제목들 = [s["제목"] for s in r["sources"]] or ["(없음)"]
    print(f"  {키:16} {r['user']:6} 근거 {', '.join(제목들)}")
    print(f"  {'':16} 답변 {r['answer'][:50]}...")
    print()

2026-11-04 14:41:39 INFO     [ml_api] 생성 종료 : {'정지 사유': '토큰 예산 소진', '토큰': 40, '대기 초': 0.0, '생성 초': 2.02}


2026-11-04 14:41:39 INFO     [ml_api] POST /ask -> 200 (머리까지 2.03초, 끝까지 2.03초, 완료)


  hr-key-003       박인사    근거 인사 평가 등급 기준
                   답변 문의하신 '인사 평가 등급은 어떻게 정합니까'에 대해 사내 규정을 확인했습니다. 평가 등급...



2026-11-04 14:41:40 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '토큰': 33, '대기 초': 0.0, '생성 초': 1.67}


2026-11-04 14:41:40 INFO     [ml_api] POST /ask -> 200 (머리까지 1.67초, 끝까지 1.67초, 완료)
  sales-key-002    이영업    근거 (없음)
                   답변 문의하신 내용은 지금 계정으로 열람할 수 있는 사내 문서에서 근거를 찾지 못했습니다. 문서...



In [43]:
%%writefile grade.py
# [3.8 채점기 grade.py (1/7)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   통과 기준 12개를 실제로 잽니다. 기능 하나에 실패 하나가 짝지어져 있습니다
#   _Abort, _post() : 공용 도우미
"""통과 기준 채점기.

1.3절의 열두 항목을 실제로 재서 통과 여부를 찍습니다.
실패를 예외로 만들지 않습니다. 한 항목이 터져도 나머지를 계속 잽니다.
"""
import json
import time
from concurrent.futures import ThreadPoolExecutor

import httpx

VALID = "hr-key-003"
SALES = "sales-key-002"
HR_Q = "인사 평가 등급은 어떻게 정합니까"
OPEN_Q = "재택근무는 며칠까지 가능합니까"


class _Abort(Exception):
    pass


def _post(base, path, body, key=VALID, timeout=180):
    headers = {"X-API-Key": key} if key else {}
    return httpx.post(base + path, json=body, headers=headers, timeout=timeout)

Overwriting grade.py


In [44]:
%%writefile -a grade.py

# [grade.py (2/7)] _stream_tokens() : SSE 를 받아 (첫 청크 시각, 청크 수)를 잼. stop_after 초가 지나면 끊음
#   run() 시작
def _stream_tokens(base, body, stop_after=None):
    """SSE 를 받습니다. stop_after 초가 지나면 끊습니다. (첫 청크 시각, 청크 수)"""
    t0 = time.perf_counter()
    first, chunks = None, 0
    try:
        with httpx.stream("POST", base + "/ask/sse", json=body,
                          headers={"X-API-Key": VALID}, timeout=180) as r:
            event = "message"
            for line in r.iter_lines():
                if line.startswith("event:"):
                    event = line.split(":", 1)[1].strip()
                elif line.startswith("data:"):
                    if event == "token":
                        chunks += 1
                        if first is None:
                            first = time.perf_counter() - t0
                if stop_after and time.perf_counter() - t0 >= stop_after:
                    raise _Abort()
    except _Abort:
        pass
    return first, chunks


def run(base):
    결과 = []

    def 기록(번호, 항목, 기준, 측정, 통과):
        결과.append((번호, 항목, 기준, str(측정), bool(통과)))

Appending to grade.py


In [45]:
%%writefile -a grade.py

# [grade.py (3/7)] run() 이어서 : 1. 헬스체크, 2. 정상 질의
    def 안전하게(번호, 항목, 기준, fn):
        try:
            측정, 통과 = fn()
        except Exception as exc:
            측정, 통과 = f"{type(exc).__name__}: {exc}"[:40], False
        기록(번호, 항목, 기준, 측정, 통과)

    # 1. 헬스체크
    def c1():
        t0 = time.perf_counter()
        r = httpx.get(base + "/health", timeout=10)
        걸림 = time.perf_counter() - t0
        return (f"{r.status_code}, {걸림:.2f}초",
                r.status_code == 200 and 걸림 < 0.5
                and r.json().get("status") == "healthy")
    안전하게(1, "헬스체크", "200, 0.5초 이내", c1)

    # 2. 정상 질의   3. 근거 표시
    상태 = {"r": None}

    def c2():
        r = _post(base, "/ask", {"question": OPEN_Q})
        상태["r"] = r
        답 = r.json().get("answer", "") if r.status_code == 200 else ""
        return f"{r.status_code}, {len(답)}자", r.status_code == 200 and len(답) >= 30
    안전하게(2, "정상 질의", "200, 답변 30자 이상", c2)

Appending to grade.py


In [46]:
%%writefile -a grade.py

# [grade.py (4/7)] run() 이어서 : 3. 근거 표시, 4. 키 없음, 5. 잘못된 키
#   잘못된 키를 한글로 쓰면 안 됩니다. HTTP 헤더는 latin-1 만 되어 클라이언트에서 UnicodeEncodeError
    def c3():
        r = 상태["r"]
        sources = r.json().get("sources", []) if r is not None else []
        return f"{len(sources)}건", len(sources) >= 1
    안전하게(3, "근거 표시", "sources 1건 이상", c3)

    # 4. 키 없음   5. 잘못된 키
    안전하게(4, "키 없음", "401", lambda: (
        (lambda s: (s, s == 401))(_post(base, "/ask", {"question": OPEN_Q},
                                        key=None).status_code)))
    # 키를 한글로 쓰지 마십시오. HTTP 헤더는 latin-1 로만 보낼 수 있어서
    # 서버에 닿기도 전에 클라이언트가 UnicodeEncodeError 로 터집니다
    안전하게(5, "잘못된 키", "401", lambda: (
        (lambda s: (s, s == 401))(_post(base, "/ask", {"question": OPEN_Q},
                                        key="wrong-key-999").status_code)))

Appending to grade.py


In [47]:
%%writefile -a grade.py

# [grade.py (5/7)] run() 이어서 : 6. 부서 밖 문서(두 계정을 함께 봄), 7. 빈 질문, 8. 너무 긴 질문, 9. 내부 오류
    # 6. 부서 밖 문서 (두 쪽을 함께 봅니다)
    def c6():
        영업 = _post(base, "/ask", {"question": HR_Q, "max_tokens": 20}, key=SALES)
        인사 = _post(base, "/ask", {"question": HR_Q, "max_tokens": 20}, key=VALID)
        영업근거 = [s["부서"] for s in 영업.json().get("sources", [])]
        인사근거 = [s["부서"] for s in 인사.json().get("sources", [])]
        샘 = [d for d in 영업근거 if d == "인사"]
        return (f"영업 계정 인사 문서 {len(샘)}건, 인사 계정 근거 {len(인사근거)}건",
                len(샘) == 0 and len(인사근거) >= 1)
    안전하게(6, "부서 밖 문서", "영업 0건 / 인사 1건 이상", c6)

    # 7. 빈 질문   8. 너무 긴 질문
    안전하게(7, "빈 질문", "422", lambda: (
        (lambda s: (s, s == 422))(_post(base, "/ask", {"question": ""}).status_code)))
    안전하게(8, "너무 긴 질문", "422", lambda: (
        (lambda s: (s, s == 422))(_post(base, "/ask",
                                        {"question": "가" * 501}).status_code)))

    # 9. 내부 오류
    def c9():
        r = _post(base, "/ask/broken", {"question": OPEN_Q})
        본문 = r.text
        샘 = [단어 for 단어 in ["Traceback", ".py", "RuntimeError", "/app"]
             if 단어 in 본문]
        return f"{r.status_code}, 새는 단어 {샘}", r.status_code == 500 and not 샘
    안전하게(9, "내부 오류", "500, 내부 정보 0곳", c9)

Appending to grade.py


In [48]:
%%writefile -a grade.py

# [grade.py (6/7)] run() 이어서 : 10. 동시 10건, 11. 스트리밍 첫 청크
    # 10. 동시 10건
    def c10():
        def 한건(i):
            return _post(base, "/ask",
                         {"question": OPEN_Q, "max_tokens": 40}).status_code
        t0 = time.perf_counter()
        with ThreadPoolExecutor(max_workers=10) as ex:
            코드들 = list(ex.map(한건, range(10)))
        걸림 = time.perf_counter() - t0
        살아있음 = httpx.get(base + "/health", timeout=10).status_code == 200
        성공 = sum(1 for c in 코드들 if c == 200)
        return (f"200 이 {성공}/10건, {걸림:.1f}초, 헬스체크 {'정상' if 살아있음 else '실패'}",
                성공 == 10 and 살아있음)
    안전하게(10, "동시 10건", "전부 200, 서버 생존", c10)

    # 11. 스트리밍 첫 청크
    def c11():
        first, chunks = _stream_tokens(base, {"question": OPEN_Q, "max_tokens": 40})
        return f"첫 청크 {first:.2f}초, 청크 {chunks}개", first is not None and first < 1.0
    안전하게(11, "스트리밍", "첫 청크 1.0초 이내", c11)

Appending to grade.py


In [49]:
%%writefile -a grade.py

# [grade.py (7/7)] run() 이어서 : 12. 취소(중간에 끊으면 만든 토큰이 절반 미만이어야 함), 결과 출력
    # 12. 취소
    def c12():
        _stream_tokens(base, {"question": OPEN_Q})            # 끝까지
        전체 = httpx.get(base + "/stats", timeout=10).json()["마지막"]["토큰"]
        _stream_tokens(base, {"question": OPEN_Q}, stop_after=0.8)
        time.sleep(1.0)                                        # 서버가 정리할 시간
        끊김 = httpx.get(base + "/stats", timeout=10).json()["마지막"]
        return (f"전체 {전체}토큰 -> 끊었을 때 {끊김['토큰']}토큰 ({끊김['정지 사유']})",
                전체 > 0 and 끊김["토큰"] < 전체 / 2)
    안전하게(12, "취소", "만든 토큰이 절반 미만", c12)

    # ===== 출력 =====
    print("=" * 78)
    print(f"{'#':>2} {'항목':<14} {'기준':<24} {'측정':<26} 판정")
    print("-" * 78)
    for 번호, 항목, 기준, 측정, 통과 in 결과:
        print(f"{번호:>2} {항목:<14} {기준:<24} {측정:<26} {'통과' if 통과 else '불통과'}")
    print("-" * 78)
    통과수 = sum(1 for *_, ok in 결과 if ok)
    print(f"  {len(결과)}개 중 {통과수}개 통과")
    if 통과수 == len(결과):
        print("  전부 통과했습니다. 제출할 수 있습니다")
    else:
        안된것 = [str(번호) for 번호, *_, ok in 결과 if not ok]
        print(f"  통과하지 못한 항목 : {', '.join(안된것)}번. 1.3절 표에서 그 줄을 보십시오")
    return 통과수 == len(결과)

Appending to grade.py


In [50]:
# [3.8] 채점 실행. 12개 모두 통과해야 합니다
ensure_qa()
import grade

importlib.reload(grade)
통과했는가 = grade.run(BASE)

2026-11-04 14:41:40 INFO     [ml_api] GET /health -> 200 (머리까지 0.00초, 끝까지 0.00초, 완료)


2026-11-04 14:41:40 INFO     [ml_api] GET /health -> 200 (머리까지 0.00초, 끝까지 0.00초, 완료)


2026-11-04 14:41:43 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '토큰': 44, '대기 초': 0.0, '생성 초': 2.22}


2026-11-04 14:41:43 INFO     [ml_api] POST /ask -> 200 (머리까지 2.23초, 끝까지 2.23초, 완료)


2026-11-04 14:41:43 WARNING  [ml_api] POST /ask -> 401 (머리까지 0.00초, 끝까지 0.00초, 완료)


2026-11-04 14:41:43 WARNING  [ml_api] POST /ask -> 401 (머리까지 0.00초, 끝까지 0.00초, 완료)


... (34 outputs left out)

2026-11-04 14:41:59 INFO     [ml_api] GET /stats -> 200 (머리까지 0.00초, 끝까지 0.00초, 완료)


2026-11-04 14:42:00 INFO     [ml_api] 생성 종료 : {'정지 사유': '밖에서 끊김', '토큰': 17, '대기 초': 0.0, '생성 초': 0.86}


2026-11-04 14:42:00 WARNING  [ml_api] POST /ask/sse -> 200 (머리까지 0.00초, 끝까지 0.87초, 중간에 끊김)


2026-11-04 14:42:01 INFO     [ml_api] GET /stats -> 200 (머리까지 0.00초, 끝까지 0.00초, 완료)
 # 항목             기준                       측정                         판정
------------------------------------------------------------------------------
 1 헬스체크           200, 0.5초 이내             200, 0.02초                 통과
 2 정상 질의          200, 답변 30자 이상           200, 130자                  통과
 3 근거 표시          sources 1건 이상            1건                         통과
 4 키 없음           401                      401                        통과
 5 잘못된 키          401                      401                        통과
 6 부서 밖 문서        영업 0건 / 인사 1건 이상         영업 계정 인사 문서 0건, 인사 계정 근거 1건 통과
 7 빈 질문           422                      422                        통과
 8 너무 긴 질문        422                      422                        통과
 9 내부 오류          500, 내부 정보 0곳            500, 새는 단어 []              통과
10 동시 10건         전부 200, 서버 생존            200 이 10/10건, 10.2초, 헬스체크 정상 통과
11 스트리밍           첫 청크 1.0초 이내 

In [51]:
# ==========================================================================
# [3.9 부하 테스트] 먼저 VRAM 어림셈으로 계산상 동시 사용자 한계를 구합니다
#   KV 캐시(토큰당) = K,V 2벌 x 층 x KV헤드 x 헤드차원 x 2바이트(fp16)
# ==========================================================================
# 상수를 박지 않습니다. 사내 GPU 와 모델이 바뀌면 이 네 줄만 고칩니다
TOTAL_VRAM_GB = float(os.environ.get("D52_VRAM_GB", "16"))
LAYERS = int(os.environ.get("D52_LAYERS", "28"))       # 7B급 모델
KV_HEADS = int(os.environ.get("D52_KV_HEADS", "4"))    # GQA 라 헤드가 적습니다
HEAD_DIM = int(os.environ.get("D52_HEAD_DIM", "128"))
SEQ = int(os.environ.get("D52_SEQ", "4096"))
WEIGHTS_GB = float(os.environ.get("D52_WEIGHTS_GB", "4.3"))   # int4 + fp16 임베딩

# 토큰 하나가 먹는 KV 캐시 : K와 V 두 벌 x 층 x 헤드 x 헤드차원 x 2바이트(fp16)
kv_per_token = 2 * LAYERS * KV_HEADS * HEAD_DIM * 2
kv_per_user_gb = kv_per_token * SEQ / 1e9

usable = TOTAL_VRAM_GB * 0.9                 # 프레임워크 버퍼와 단편화를 감안
weights = WEIGHTS_GB * 1.15                  # 활성화와 여유
계산상_한계 = int((usable - weights) / kv_per_user_gb)

print(f"GPU 전체        : {TOTAL_VRAM_GB:.0f} GB   (쓸 수 있는 몫 {usable:.1f} GB)")
print(f"가중치          : {weights:.2f} GB")
print(f"사용자 한 명당  : {kv_per_user_gb:.3f} GB  (시퀀스 {SEQ} 기준)")
print(f"계산상 동시 사용자 한계 : {계산상_한계}명")

GPU 전체        : 16 GB   (쓸 수 있는 몫 14.4 GB)
가중치          : 4.94 GB
사용자 한 명당  : 0.235 GB  (시퀀스 4096 기준)
계산상 동시 사용자 한계 : 40명


In [52]:
# [3.9 실측 (1/2)] 동시 요청 수를 늘려 가며 약속한 6초 안에 답하는 최대 인원을 잽니다
ensure_qa()
import statistics
from concurrent.futures import ThreadPoolExecutor

# 약속한 응답 시간. 기술이 아니라 서비스가 정하는 값이라 환경 변수로 뺐습니다
목표초 = float(os.environ.get("D52_TARGET_SECONDS", "6"))

2026-11-04 14:42:01 INFO     [ml_api] GET /health -> 200 (머리까지 0.00초, 끝까지 0.00초, 완료)


In [53]:
# [3.9 실측 (2/2)] 부하(n) : 동시에 n건 요청을 보내 중앙값/최대 응답 시간, 실패 수를 잽니다
def 부하(n, 질문="재택근무는 며칠까지 가능합니까"):
    def 한건(_):
        t0 = time.perf_counter()
        r = httpx.post(BASE + "/ask", json={"question": 질문},
                       headers={"X-API-Key": "dev-key-001"}, timeout=180)
        return time.perf_counter() - t0, r.status_code

    with ThreadPoolExecutor(max_workers=n) as ex:
        결과 = list(ex.map(한건, range(n)))
    시간들 = sorted(t for t, _ in 결과)
    실패 = sum(1 for _, c in 결과 if c != 200)
    return {"동시": n, "중앙값": statistics.median(시간들),
            "최대": 시간들[-1], "실패": 실패}


표 = [부하(n) for n in [1, 2, 4, 8]]

print(f"{'동시':>4} {'중앙값':>8} {'최대':>8} {'실패':>5}  {'목표 ' + str(int(목표초)) + '초':>10}")
print("-" * 46)
for 행 in 표:
    판정 = "통과" if 행["최대"] <= 목표초 and 행["실패"] == 0 else "초과"
    print(f"{행['동시']:>4} {행['중앙값']:>7.1f}초 {행['최대']:>7.1f}초 "
          f"{행['실패']:>5} {판정:>10}")

통과한것 = [행["동시"] for 행 in 표 if 행["최대"] <= 목표초 and 행["실패"] == 0]
실측_한계 = max(통과한것) if 통과한것 else 0
print()
print(f"실측 한계 : 동시 {실측_한계}명까지 {int(목표초)}초 안에 답합니다")

2026-11-04 14:42:04 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '토큰': 44, '대기 초': 0.0, '생성 초': 2.22}


2026-11-04 14:42:04 INFO     [ml_api] POST /ask -> 200 (머리까지 2.22초, 끝까지 2.22초, 완료)


2026-11-04 14:42:06 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '토큰': 44, '대기 초': 0.0, '생성 초': 2.23}


2026-11-04 14:42:06 INFO     [ml_api] POST /ask -> 200 (머리까지 2.23초, 끝까지 2.23초, 완료)


2026-11-04 14:42:06 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '토큰': 44, '대기 초': 0.0, '생성 초': 2.23}


2026-11-04 14:42:06 INFO     [ml_api] POST /ask -> 200 (머리까지 2.23초, 끝까지 2.23초, 완료)


... (21 outputs left out)

2026-11-04 14:42:19 INFO     [ml_api] POST /ask -> 200 (머리까지 8.89초, 끝까지 8.89초, 완료)


2026-11-04 14:42:19 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '토큰': 44, '대기 초': 6.66, '생성 초': 2.23}


2026-11-04 14:42:19 INFO     [ml_api] POST /ask -> 200 (머리까지 8.89초, 끝까지 8.89초, 완료)


  동시      중앙값       최대    실패       목표 6초
----------------------------------------------
   1     2.2초     2.2초     0         통과
   2     2.3초     2.3초     0         통과
   4     3.4초     4.5초     0         통과
   8     5.6초     8.9초     0         초과

실측 한계 : 동시 4명까지 6초 안에 답합니다


In [54]:
# [3.9] 계산값(VRAM)과 실측값(응답 시간) 중 작은 쪽이 서비스 한계입니다
from app.generation import GPU_SLOTS

print(f"{'한계':<22} {'값':>8}  무엇이 정합니까")
print("-" * 70)
print(f"{'계산 (VRAM)':<22} {계산상_한계:>6}명  모델 크기, 컨텍스트 길이, GPU 용량")
print(f"{'실측 (응답 시간)':<22} {실측_한계:>6}명  생성 속도와 동시 통과 슬롯 수 ({GPU_SLOTS}칸)")
print("-" * 70)
print(f"{'서비스 한계':<22} {min(계산상_한계, 실측_한계):>6}명  둘 중 작은 쪽")

한계                            값  무엇이 정합니까
----------------------------------------------------------------------
계산 (VRAM)                  40명  모델 크기, 컨텍스트 길이, GPU 용량
실측 (응답 시간)                  4명  생성 속도와 동시 통과 슬롯 수 (2칸)
----------------------------------------------------------------------
서비스 한계                      4명  둘 중 작은 쪽


In [55]:
# [3.9] 슬롯 값(환경 변수 D52_GPU_SLOTS)을 2 -> 8 로 바꿨을 때의 변화
# 슬롯만 늘려 다시 띄웁니다. 코드는 한 줄도 고치지 않습니다
os.environ["D52_GPU_SLOTS"] = "8"
print(serve(TARGET))

늘린뒤 = 부하(8)
print(f"  슬롯 8칸일 때 동시 8명 : 최대 {늘린뒤['최대']:.1f}초 "
      f"(실패 {늘린뒤['실패']}건)")

os.environ["D52_GPU_SLOTS"] = "2"          # 원래대로 돌려 둡니다
print(serve(TARGET))

2026-11-04 14:42:19 INFO     [ml_api] 서버를 내립니다


2026-11-04 14:42:20 INFO     [ml_api] 사내 문서를 읽습니다


2026-11-04 14:42:20 INFO     [ml_api] 준비 완료. 문서 7건


기동 : app.main:app -> http://127.0.0.1:8453


2026-11-04 14:42:22 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '토큰': 44, '대기 초': 0.01, '생성 초': 2.23}


2026-11-04 14:42:22 INFO     [ml_api] POST /ask -> 200 (머리까지 2.24초, 끝까지 2.24초, 완료)


... (15 outputs left out)

2026-11-04 14:42:22 INFO     [ml_api] 서버를 내립니다


2026-11-04 14:42:22 INFO     [ml_api] 사내 문서를 읽습니다


2026-11-04 14:42:22 INFO     [ml_api] 준비 완료. 문서 7건


기동 : app.main:app -> http://127.0.0.1:8453


In [56]:
%%writefile frontend/app.py
# [3.10 화면 frontend/app.py (1/5)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   스트리밍으로 답을 받고 근거 문서를 함께 보여 줍니다
#   실행 (터미널, 서버가 떠 있는 동안):
#     D52_API_BASE=http://127.0.0.1:8452 .venv-wsl312/bin/python -m streamlit run frontend/app.py --server.port 8502
# 실습 : 화면 (API 키 입력, 스트리밍 답, 근거 표시)
"""사내 문서 질의응답 화면.

실행 : streamlit run frontend/app.py
참고 : 인증과 화면 노트의 화면 장, 챗봇 프로젝트 부록 4장.
"""
import json
import os

import httpx
import streamlit as st

API_BASE = os.environ.get("D52_API_BASE", "http://127.0.0.1:8452")

st.set_page_config(page_title="사내 문서 질의응답", page_icon="📄", layout="centered")

Overwriting frontend/app.py


In [57]:
%%writefile -a frontend/app.py

# [frontend/app.py (2/5)] stream_answer() : SSE 를 받아 토큰을 하나씩 내보내고, 근거는 st.session_state 에 담습니다
def stream_answer(question, api_key):
    """SSE 를 받아 토큰을 하나씩 스트리밍으로 보냅니다. 근거는 st.session_state 에 담습니다."""
    st.session_state["마지막 근거"] = []
    try:
        with httpx.stream("POST", f"{API_BASE}/ask/sse",
                          json={"question": question},
                          headers={"X-API-Key": api_key}, timeout=180) as r:
            if r.status_code == 401:
                yield "인증에 실패했습니다. 사이드바의 API 키를 확인해 주십시오."
                return
            if r.status_code >= 400:
                yield f"요청이 거부되었습니다 (HTTP {r.status_code})."
                return
            event = "message"
            for line in r.iter_lines():
                if line.startswith("event:"):
                    event = line.split(":", 1)[1].strip()
                elif line.startswith("data:"):
                    data = json.loads(line.split(":", 1)[1])
                    if event == "start":
                        st.session_state["마지막 근거"] = data.get("sources", [])
                    elif event == "token":
                        yield data["text"]
                    elif event == "error":
                        yield f"\n\n({data.get('error', '오류')})"
    except httpx.ConnectError:
        yield "서버에 연결할 수 없습니다. 서버가 떠 있는지 확인해 주십시오."

Appending to frontend/app.py


In [58]:
%%writefile -a frontend/app.py

# [frontend/app.py (3/5)] 사이드바(API 키 입력, 서버 상태) + 본문 제목
# ===== 사이드바 =====
with st.sidebar:
    st.header("설정")
    # 1) 사이드바에 API 키 입력란을 만듭니다. 기본값은 실습용 키 dev-key-001
    api_key = st.text_input("API 키", value="dev-key-001", type="password")

    try:
        health = httpx.get(f"{API_BASE}/health", timeout=3).json()
        if health.get("status") == "healthy":
            st.success(f"서버 연결됨 (문서 {health.get('문서 수', 0)}건)")
        else:
            st.warning("서버가 아직 준비 중입니다")
    except Exception:
        st.error("서버에 연결할 수 없습니다")

    if st.button("대화 지우기"):
        st.session_state["기록"] = []
        st.rerun()

# ===== 본문 =====
st.title("사내 문서 질의응답")
st.caption("사내망 안에서만 동작합니다. 계정에 따라 볼 수 있는 문서가 다릅니다.")

if "기록" not in st.session_state:
    st.session_state["기록"] = []

Appending to frontend/app.py


In [59]:
%%writefile -a frontend/app.py

# [frontend/app.py (4/5)] 대화 기록 표시 + 질문 입력란
for 말 in st.session_state["기록"]:
    with st.chat_message(말["role"]):
        st.write(말["content"])
        for s in 말.get("sources", []):
            st.caption(f"근거 : {s['제목']} ({s['부서']})")

질문 = st.chat_input("사내 규정에 대해 물어보십시오")

Appending to frontend/app.py


In [60]:
%%writefile -a frontend/app.py

# [frontend/app.py (5/5)] 질문이 들어오면 st.write_stream 으로 스트리밍 답을 쓰고, 근거(제목, 부서)를 답 아래에 표시
if 질문:
    st.session_state["기록"].append({"role": "user", "content": 질문})
    with st.chat_message("user"):
        st.write(질문)

    with st.chat_message("assistant"):
        # 1) stream_answer 제너레이터를 st.write_stream 에 넘깁니다. 이어 붙인 전체 답이 돌아옵니다
        답 = st.write_stream(stream_answer(질문, api_key))
        # 2) start 이벤트에서 담아 둔 근거를 꺼내 제목과 부서를 답 아래에 표시합니다
        근거 = st.session_state.get("마지막 근거", [])
        for s in 근거:
            st.caption(f"근거 : {s['제목']} ({s['부서']})")

    st.session_state["기록"].append(
        {"role": "assistant", "content": 답, "sources": 근거})

Appending to frontend/app.py


In [61]:
# [3.10] 화면 파일의 문법과 필수 요소(환경 변수 주소, 스트리밍, 근거 표시)를 점검합니다
# 코랩에서 화면 대신 확인할 수 있는 것 : 문법이 맞는지, 그리고 서버 주소가 박혀 있지 않은지
import ast

소스 = (WORK / "frontend" / "app.py").read_text(encoding="utf-8")
try:
    ast.parse(소스)
    print("  문법 : 통과")
except SyntaxError as e:
    print(f"  문법 : 실패 {e.lineno}행 {e.msg}")

print(f"  환경 변수로 주소를 읽는가 : {'D52_API_BASE' in 소스}")
print(f"  스트리밍을 쓰는가        : {'/ask/sse' in 소스}")
print(f"  근거를 표시하는가        : {'근거' in 소스}")
print(f"  줄 수                    : {len(소스.splitlines())}줄")

  문법 : 통과
  환경 변수로 주소를 읽는가 : True
  스트리밍을 쓰는가        : True
  근거를 표시하는가        : True
  줄 수                    : 93줄


In [62]:
# ==========================================================================
# [4. 선택 트랙 : 정형 데이터 예측 서비스]
#   같은 조립 순서(스키마 -> 추론 함수 -> 엔드포인트 -> 422 -> 인증)를 가벼운 모델로 한 번 더 돕니다
# ==========================================================================
%pip install -q "scikit-learn>=1.9,<2" "joblib>=1.6,<2"

Note: you may need to restart the kernel to use updated packages.


In [63]:
# [4] 설치된 버전 확인
from importlib.metadata import version

print("scikit-learn", version("scikit-learn"))
print("joblib      ", version("joblib"))

scikit-learn 1.9.1
joblib       1.6.0


In [64]:
# [4. 데이터 (1/2)] load_table() : California Housing 데이터를 읽습니다
#   다운로드가 막히면 같은 모양의 표를 만듭니다
import numpy as np

FEATURES = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
            "Population", "AveOccup", "Latitude", "Longitude"]


def load_table(seed=42):
    """주택 가격 표를 읽습니다. 다운로드가 막히면 같은 모양의 표를 만듭니다."""
    try:
        from sklearn.datasets import fetch_california_housing
        d = fetch_california_housing()
        return d.data, d.target, "다운로드"
    except Exception as exc:
        print(f"  다운로드 실패 ({type(exc).__name__}). 같은 모양의 표를 만듭니다")
        rng = np.random.default_rng(seed)
        n = 20640
        X = np.column_stack([
            rng.gamma(2.0, 1.8, n) + 0.5,              # MedInc
            rng.integers(1, 52, n).astype(float),       # HouseAge
            rng.normal(5.4, 1.2, n).clip(1, 20),        # AveRooms
            rng.normal(1.1, 0.2, n).clip(0.5, 5),       # AveBedrms
            rng.gamma(2.0, 700, n).clip(5, 35000),      # Population
            rng.normal(3.1, 0.8, n).clip(1, 15),        # AveOccup
            rng.uniform(32.5, 42.0, n),                 # Latitude
            rng.uniform(-124.3, -114.3, n),             # Longitude
        ])
        y = (0.42 * X[:, 0] - 0.004 * X[:, 1] + 0.05 * X[:, 2]
             - 0.12 * X[:, 5] - 0.09 * (X[:, 6] - 35.6)
             + rng.normal(0, 0.45, n) + 0.9).clip(0.15, 5.0)
        return X, y, "합성"

In [65]:
# [4. 데이터 (2/2)] 표를 읽고 크기와 타겟 범위를 확인합니다
X, y, 출처 = load_table()
print(f"출처       : {출처}")
print(f"표 크기    : {X.shape[0]:,}행 {X.shape[1]}열")
print(f"타겟 범위  : {y.min():.2f} ~ {y.max():.2f}  (10만 달러 단위)")

출처       : 다운로드
표 크기    : 20,640행 8열
타겟 범위  : 0.15 ~ 5.00  (10만 달러 단위)


In [66]:
# [4.1 학습] StandardScaler + MLPRegressor 파이프라인을 학습하고 테스트 MAE 를 잽니다
import json
import time

import joblib
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

파이프 = Pipeline([
    ("scaler", StandardScaler()),                       # 전처리
    ("mlp", MLPRegressor(hidden_layer_sizes=(64, 32),   # 모델
                         max_iter=60, random_state=42,
                         early_stopping=True, n_iter_no_change=8)),
])

t0 = time.perf_counter()
파이프.fit(X_train, y_train)
학습시간 = time.perf_counter() - t0

예측 = 파이프.predict(X_test)
mae = float(np.abs(예측 - y_test).mean())

print(f"학습 시간 : {학습시간:.1f}초")
print(f"테스트 MAE : {mae:.3f}  (10만 달러 단위)  = 약 ${mae * 100000:,.0f}")

학습 시간 : 4.1초
테스트 MAE : 0.367  (10만 달러 단위)  = 약 $36,660


/mnt/c/EPA1st/model-serving-course/.venv-wsl312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (60) reached and the optimization hasn't converged yet.
  warnings.warn(


In [67]:
# [4.2 저장] 모델(joblib) + 메타 정보(json)를 models/ 에 저장합니다
MODEL_PATH = WORK / "models" / "housing_pipeline.joblib"
META_PATH = WORK / "models" / "housing_meta.json"

joblib.dump(파이프, MODEL_PATH)

META = {
    "features": FEATURES,
    "target": "중위 주택 가격 (10만 달러 단위)",
    "mae": round(mae, 4),
    "학습 행 수": int(X_train.shape[0]),
    "데이터 출처": 출처,
    "sklearn": version("scikit-learn"),
}
META_PATH.write_text(json.dumps(META, ensure_ascii=False, indent=1), encoding="utf-8")

print(f"모델 : {MODEL_PATH.name}  ({MODEL_PATH.stat().st_size / 1024:.1f} KB)")
print(f"메타 : {META_PATH.name}")
print(json.dumps(META, ensure_ascii=False, indent=1))

모델 : housing_pipeline.joblib  (72.6 KB)
메타 : housing_meta.json
{
 "features": [
  "MedInc",
  "HouseAge",
  "AveRooms",
  "AveBedrms",
  "Population",
  "AveOccup",
  "Latitude",
  "Longitude"
 ],
 "target": "중위 주택 가격 (10만 달러 단위)",
 "mae": 0.3666,
 "학습 행 수": 16512,
 "데이터 출처": "다운로드",
 "sklearn": "1.9.1"
}


In [68]:
%%writefile app/tabular_api.py
# [4.3 예측 서버 app/tabular_api.py (1/5)] 파일 시작 (이후 셀은 -a 로 이어 씁니다)
#   import 와 설정값
"""정형 데이터 예측 서비스 (선택 트랙).

3장의 질의응답 서버와 구조가 같습니다. 모델이 가벼워서 스트리밍도
동시 통과 제한도 필요 없습니다. 그래서 층이 몇 개 빠졌을 뿐입니다.
"""
import json
import os
from pathlib import Path

import joblib
import numpy as np
from contextlib import asynccontextmanager

from fastapi import Depends, FastAPI, HTTPException
from pydantic import BaseModel, Field

from app.auth import verify_api_key
from app.error_handlers import register_error_handlers
from app.logger_config import setup_logger
from app.middleware import RequestLoggingMiddleware

logger = setup_logger("ml_api")

MODEL_PATH = Path(os.environ.get("D52_TABULAR_MODEL", "models/housing_pipeline.joblib"))
META_PATH = Path(os.environ.get("D52_TABULAR_META", "models/housing_meta.json"))

STATE = {"준비됨": False, "mae": None}
MODEL = {"pipe": None, "features": []}

Overwriting app/tabular_api.py


In [69]:
%%writefile -a app/tabular_api.py

# [app/tabular_api.py (2/5)] PriceRequest : 주택 정보. 범위를 벗어난 값은 여기서 422 로 막힙니다
class PriceRequest(BaseModel):
    """주택 정보. 범위를 벗어난 값은 여기서 막힙니다."""
    MedInc: float = Field(..., gt=0, le=20, description="중위 소득")
    HouseAge: float = Field(..., ge=0, le=100, description="주택 연식(년)")
    AveRooms: float = Field(..., gt=0, le=50, description="평균 방 수")
    AveBedrms: float = Field(..., gt=0, le=20, description="평균 침실 수")
    Population: float = Field(..., gt=0, le=50000, description="인구")
    AveOccup: float = Field(..., gt=0, le=20, description="평균 거주 인원")
    Latitude: float = Field(..., ge=32, le=42, description="위도 (캘리포니아 범위)")
    Longitude: float = Field(..., ge=-125, le=-114, description="경도 (캘리포니아 범위)")

    model_config = {
        "json_schema_extra": {
            "examples": [{"MedInc": 3.5, "HouseAge": 25.0, "AveRooms": 5.0,
                          "AveBedrms": 1.0, "Population": 1500.0, "AveOccup": 3.0,
                          "Latitude": 37.5, "Longitude": -122.0}]
        }
    }

Appending to app/tabular_api.py


In [70]:
%%writefile -a app/tabular_api.py

# [app/tabular_api.py (3/5)] PriceResponse : 예측 응답 규격
class PriceResponse(BaseModel):
    success: bool = True
    predicted_price: float = Field(description="예측 가격 (10만 달러 단위)")
    predicted_price_usd: int = Field(description="예측 가격 (달러)")
    user: str = Field(description="인증된 사용자")
    model_mae: float | None = Field(default=None, description="이 모델의 테스트 MAE")

Appending to app/tabular_api.py


In [71]:
%%writefile -a app/tabular_api.py

# [app/tabular_api.py (4/5)] lifespan() : 모델 로드 / GET /health
#   모델 로드 실패를 예외로 만들지 않습니다. 서버는 뜨고 헬스체크가 loading 을 알립니다
@asynccontextmanager
async def lifespan(app: FastAPI):
    if MODEL_PATH.exists() and META_PATH.exists():
        MODEL["pipe"] = joblib.load(MODEL_PATH)
        meta = json.loads(META_PATH.read_text(encoding="utf-8"))
        MODEL["features"] = meta["features"]
        STATE["mae"] = meta.get("mae")
        STATE["준비됨"] = True
        logger.info(f"모델 로드 완료. 피처 {len(MODEL['features'])}개, MAE {STATE['mae']}")
    else:
        # 실패를 예외로 만들지 않습니다. 서버는 뜨고 헬스체크가 loading 을 알립니다
        logger.error(f"모델 파일이 없습니다 : {MODEL_PATH}")
    yield
    STATE["준비됨"] = False


app = FastAPI(title="주택 가격 예측", version="1.0.0", lifespan=lifespan)
app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)


@app.get("/health", tags=["System"])
async def health():
    return {"status": "healthy" if STATE["준비됨"] else "loading", **STATE}

Appending to app/tabular_api.py


In [72]:
%%writefile -a app/tabular_api.py

# [app/tabular_api.py (5/5)] POST /predict
#   추론이 1밀리초 미만이라 def 로 둡니다
#   피처 순서를 모델이 학습한 순서에 맞춥니다 (어긋나면 조용히 틀림)
# 추론이 1밀리초 미만이라 def 로 둡니다. 비동기 처리 노트의 판단 기준 그대로입니다
@app.post("/predict", response_model=PriceResponse, tags=["Prediction"])
def predict(req: PriceRequest, user: dict = Depends(verify_api_key)):
    if MODEL["pipe"] is None:
        raise HTTPException(status_code=503, detail="모델이 아직 로드되지 않았습니다.")

    데이터 = req.model_dump()
    # 피처 순서를 모델이 학습한 순서에 맞춥니다. 순서가 어긋나면 조용히 틀립니다
    행 = np.array([[데이터[이름] for 이름 in MODEL["features"]]], dtype=np.float64)
    값 = float(MODEL["pipe"].predict(행)[0])
    값 = max(값, 0.0)

    return PriceResponse(
        predicted_price=round(값, 4),
        predicted_price_usd=int(값 * 100000),
        user=user["사용자"],
        model_mae=STATE["mae"],
    )

Appending to app/tabular_api.py


In [73]:
# [4.4 띄우고 확인] 예측 서버를 띄우고 헬스체크
print(serve("app.tabular_api:app"))
print(httpx.get(BASE + "/health", timeout=10).json())

2026-11-04 14:42:41 INFO     [ml_api] 서버를 내립니다


2026-11-04 14:42:41 INFO     [ml_api] 모델 로드 완료. 피처 8개, MAE 0.3666


기동 : app.tabular_api:app -> http://127.0.0.1:8453
2026-11-04 14:42:41 INFO     [ml_api] GET /health -> 200 (머리까지 0.01초, 끝까지 0.01초, 완료)


{'status': 'healthy', '준비됨': True, 'mae': 0.3666}


In [74]:
# [4.4] 정상 요청 예측 + 소득만 올렸을 때 예측이 따라 올라가는지 확인
정상요청 = {"MedInc": 3.5, "HouseAge": 25.0, "AveRooms": 5.0, "AveBedrms": 1.0,
          "Population": 1500.0, "AveOccup": 3.0, "Latitude": 37.5, "Longitude": -122.0}
헤더 = {"X-API-Key": "dev-key-001"}

r = httpx.post(BASE + "/predict", json=정상요청, headers=헤더, timeout=30)
print(f"  상태 {r.status_code}")
print(f"  예측 ${r.json()['predicted_price_usd']:,}  (MAE ${r.json()['model_mae'] * 100000:,.0f})")

# 소득만 올려 봅니다. 예측이 따라 올라가야 모델이 말이 됩니다
고소득 = dict(정상요청, MedInc=9.0)
r2 = httpx.post(BASE + "/predict", json=고소득, headers=헤더, timeout=30)
print(f"  소득 3.5 -> 9.0 : ${r.json()['predicted_price_usd']:,} "
      f"-> ${r2.json()['predicted_price_usd']:,}")

2026-11-04 14:42:41 INFO     [ml_api] POST /predict -> 200 (머리까지 0.00초, 끝까지 0.00초, 완료)


  상태 200
  예측 $199,217  (MAE $36,660)
2026-11-04 14:42:41 INFO     [ml_api] POST /predict -> 200 (머리까지 0.00초, 끝까지 0.00초, 완료)
  소득 3.5 -> 9.0 : $199,217 -> $352,614


In [75]:
# [4.4] 실패 네 가지(키 없음, 필드 누락, 위도 범위 초과, 소득 음수)가 401/422 로 나가는지 확인
# 실패 네 가지. 3장 채점기와 같은 생각입니다
시험 = [
    ("키 없음", {}, 정상요청, 401),
    ("필드 누락", 헤더, {"MedInc": 3.5}, 422),
    ("위도 범위 초과", 헤더, dict(정상요청, Latitude=50.0), 422),
    ("소득 음수", 헤더, dict(정상요청, MedInc=-1.0), 422),
]

for 이름, h, body, 기대 in 시험:
    코드 = httpx.post(BASE + "/predict", json=body, headers=h, timeout=30).status_code
    print(f"  {이름:14} -> {코드}  기대 {기대}  {'통과' if 코드 == 기대 else '불통과'}")

print(f"  헬스체크        -> {httpx.get(BASE + '/health', timeout=10).status_code}  서버 생존")

2026-11-04 14:42:41 WARNING  [ml_api] POST /predict -> 401 (머리까지 0.00초, 끝까지 0.00초, 완료)


  키 없음           -> 401  기대 401  통과
2026-11-04 14:42:41 WARNING  [ml_api] POST /predict -> 422 (머리까지 0.00초, 끝까지 0.00초, 완료)
  필드 누락          -> 422  기대 422  통과


2026-11-04 14:42:41 WARNING  [ml_api] POST /predict -> 422 (머리까지 0.00초, 끝까지 0.00초, 완료)


  위도 범위 초과       -> 422  기대 422  통과


2026-11-04 14:42:41 WARNING  [ml_api] POST /predict -> 422 (머리까지 0.00초, 끝까지 0.00초, 완료)
  소득 음수          -> 422  기대 422  통과


2026-11-04 14:42:41 INFO     [ml_api] GET /health -> 200 (머리까지 0.00초, 끝까지 0.00초, 완료)
  헬스체크        -> 200  서버 생존


In [76]:
# [4.4] 서버를 내립니다
print(unserve())

정지


In [77]:
# ==========================================================================
# [5. 오늘 만든 파일] 파일별 줄 수와 역할
#
# [정리]
#   권한은 검색 '전에' 건다                  -> retrieve() 첫 줄
#   모르면 불합격                            -> service.resolve_docs, in_scope
#   401(키) / 422(입력) / 500(내부, 정보 안 샘) -> auth.py, schemas.py, error_handlers.py
#   스트리밍과 취소                          -> /ask/sse, is_disconnected()
#   동시 처리 제한과 예산                    -> CapacityLimiter, 토큰/시간 예산
#   용량은 숫자 두 개로 본다                 -> VRAM 계산 40명 vs 실측 4명 -> 4명
#   설정은 환경 변수로                       -> D52_*
# ==========================================================================
print(f"{'파일':<34} {'줄':>5}  역할")
print("-" * 76)

목록 = [
    ("app/inference.py", "검색과 생성. 진짜 모델로 바꾸는 자리"),
    ("app/generation.py", "예산, 이탈, 동시 통과 제한"),
    ("app/schemas.py", "요청과 응답의 규격"),
    ("app/service.py", "누가 무엇을 볼 수 있는지"),
    ("app/main.py", "엔드포인트 조립"),
    ("app/auth.py", "API 키 -> 사용자와 부서"),
    ("app/logger_config.py", "로깅 설정"),
    ("app/error_handlers.py", "글로벌 에러 핸들러"),
    ("app/middleware.py", "요청 로깅 (순수 ASGI)"),
    ("frontend/app.py", "화면"),
    ("grade.py", "통과 기준 채점기"),
    ("data/사내문서.json", "부서가 붙은 사내 문서"),
]

for 이름, 역할 in 목록:
    p = WORK / 이름
    줄 = len(p.read_text(encoding="utf-8").splitlines()) if p.exists() else 0
    print(f"{이름:<34} {줄:>5}  {역할}")

파일                                     줄  역할
----------------------------------------------------------------------------
app/inference.py                      82  검색과 생성. 진짜 모델로 바꾸는 자리


app/generation.py                     82  예산, 이탈, 동시 통과 제한
app/schemas.py                        50  요청과 응답의 규격
app/service.py                        34  누가 무엇을 볼 수 있는지
app/main.py                          128  엔드포인트 조립
app/auth.py                           35  API 키 -> 사용자와 부서


app/logger_config.py                  19  로깅 설정
app/error_handlers.py                 22  글로벌 에러 핸들러
app/middleware.py                     56  요청 로깅 (순수 ASGI)
frontend/app.py                       93  화면
grade.py                             171  통과 기준 채점기
data/사내문서.json                        37  부서가 붙은 사내 문서
